# Notebook 14 — Semantic hierarchy and Alert-Semantic Vulnerability Graph

**Purpose.** Create the validation-only binary–family–fine taxonomy, three
predefined operational cost profiles, and the directed Alert-Semantic
Vulnerability Graph (ASVG) that all later SABER-IDS experiments use.

**Run before:** Notebooks 15–18.

**Primary outputs**

- `results/saber/14_risk_graph/ciciot2023_taxonomy.csv`
- `results/saber/14_risk_graph/cost_profiles.json`
- `results/saber/14_risk_graph/asvg_edges_by_cost_profile.csv`
- `results/saber/14_risk_graph/asvg_edges_robust.csv`
- graph diagnostics, plots, and a reproducibility manifest

**Integrity rule.** All graph evidence is constructed from the frozen
**validation** split. Test predictions are not read in this notebook.

In [ ]:
# Colab/repository bootstrap
from pathlib import Path
import os, sys, json, subprocess, platform

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# Override with %env SABER_REPO=/your/path if your repository is elsewhere.
candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError(
        "Repository not found. Set SABER_REPO or edit the candidate path."
    )
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())

In [ ]:
# Install only the small SABER extension requirements.
# The original repository requirements must already be installed.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-saber.txt"],
        check=True,
    )

In [ ]:
import yaml
from src.saber.adapters import SaberRepo

repo = SaberRepo.discover(REPO)
with open(REPO / "config" / "saber.yaml", "r", encoding="utf-8") as handle:
    SABER_CFG = yaml.safe_load(handle)

OUTPUT_ROOT = repo.output_root
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("SABER output root:", OUTPUT_ROOT)
print("Config branch:", SABER_CFG["project"]["branch"])

## 1. Load the frozen teacher and validation split

The bridge tries the historical repository API. If it cannot infer your loader
factory or anchor checkpoint unambiguously, define the five objects shown in the
cell comments using the corresponding cells from the completed notebooks 09–13.
Do not silently choose between multiple checkpoints.

In [ ]:
# --- Manual bridge (full, idempotent) ---
import numpy as np, torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler

from src.config import CFG, PATHS, set_all_seeds
from src import data as D, models as M

SEED = int(CFG['anchor_seed'])
set_all_seeds(SEED)

# 1. Dataset + frozen primary split (reuse if already loaded this session)
if 'df' not in globals() or 'splits' not in globals():
    df = D.clean(D.load_raw('ciciot2023', subsample=True, seed=SEED), 'ciciot2023')
    splits = D.temporal_within_capture_split(df, seed=SEED)

# 2. Detect the label column; features = numeric columns only
str_cols = [c for c in df.columns if not np.issubdtype(df[c].dtype, np.number)]
print('non-numeric columns:', {c: int(df[c].nunique()) for c in str_cols})
LABEL_COL = next(c for c in str_cols if 'BenignTraffic' in set(df[c].astype(str).unique()))
feat_cols = [c for c in df.columns if c not in str_cols]
print('label column:', LABEL_COL, '| numeric features:', len(feat_cols))

# 3. Encoder + TRAIN-only scaler
le = LabelEncoder().fit(df[LABEL_COL].to_numpy())
scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
CLASS_NAMES = list(le.classes_)
assert len(CLASS_NAMES) == 34, f"expected 34 classes, got {len(CLASS_NAMES)}"

def _loader(idx, shuffle):
    X = torch.tensor(scaler.transform(df.loc[idx, feat_cols].to_numpy(np.float32)))
    y = torch.tensor(le.transform(df.loc[idx, LABEL_COL].to_numpy()), dtype=torch.long)
    return DataLoader(TensorDataset(X, y), batch_size=1024, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(SEED))

TRAIN_LOADER = _loader(splits['train'], True)
VAL_LOADER   = _loader(splits['val'],   False)
TEST_LOADER  = _loader(splits['test'],  False)

# 4. Frozen anchor checkpoint (strict load; channels inferred from its shapes)
ckpt_path = PATHS.models() / f'cnn1d_M0_seed{SEED}.pt'
sd = torch.load(ckpt_path, map_location='cpu')['state_dict']
ch = (int(sd['conv.0.weight'].shape[0]), int(sd['conv.3.weight'].shape[0]))
MODEL = M.build('cnn1d', in_dim=len(feat_cols), n_classes=len(CLASS_NAMES), channels=ch)
MODEL.load_state_dict(sd); MODEL.eval()

print('bridge OK | train/val/test:',
      len(splits['train']), len(splits['val']), len(splits['test']),
      '| features:', len(feat_cols), '| classes:', len(CLASS_NAMES), '| channels:', ch)

In [ ]:
# --- 4 (final): load the frozen anchor checkpoint, exact repo naming ---
from pathlib import Path

ckpt_path = Path('models/ciciot2023') / f'ciciot2023__cnn1d__M0__seed{SEED}.pt'
assert ckpt_path.exists(), f"not found: {ckpt_path}"

sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)['state_dict']
ch = (int(sd['conv.0.weight'].shape[0]), int(sd['conv.3.weight'].shape[0]))
MODEL = M.build('cnn1d', in_dim=len(feat_cols), n_classes=len(CLASS_NAMES), channels=ch)
MODEL.load_state_dict(sd); MODEL.eval()

print('anchor loaded from:', ckpt_path)
print('bridge OK | train/val/test:',
      len(splits['train']), len(splits['val']), len(splits['test']),
      '| features:', len(feat_cols), '| classes:', len(CLASS_NAMES), '| channels:', ch)

In [ ]:
# Repository bridge: auto-discovery first, explicit override second.
#
# If auto-discovery fails, set these objects using the same loader/model
# construction cells from the completed Computer Networks notebooks:
#   TRAIN_LOADER = ...
#   VAL_LOADER = ...
#   TEST_LOADER = ...
#   MODEL = ...
#   CLASS_NAMES = [...]
#
# MODEL must be the uncompressed CNN1D anchor and loaders must use the frozen
# train/validation/test split.

import torch
from src.saber.adapters import (
    auto_discover_loaders,
    auto_build_cnn,
    discover_anchor_checkpoint,
    infer_class_names_from_results,
    unpack_batch,
)

TRAIN_LOADER = globals().get("TRAIN_LOADER")
VAL_LOADER = globals().get("VAL_LOADER")
TEST_LOADER = globals().get("TEST_LOADER")
MODEL = globals().get("MODEL")
CLASS_NAMES = globals().get("CLASS_NAMES")

if any(obj is None for obj in (TRAIN_LOADER, VAL_LOADER, TEST_LOADER)):
    TRAIN_LOADER, VAL_LOADER, TEST_LOADER, _DATA_BUNDLE = auto_discover_loaders(repo)

first_batch = next(iter(VAL_LOADER))
x0, y0, env0 = unpack_batch(first_batch)
raw_example = x0[: min(8, len(x0))].float()

if CLASS_NAMES is None:
    CLASS_NAMES = infer_class_names_from_results(repo)

if MODEL is None:
    checkpoint = discover_anchor_checkpoint(repo)
    MODEL, MODEL_FACTORY_ERRORS = auto_build_cnn(
        n_features=int(x0.shape[-1]),
        n_classes=len(CLASS_NAMES),
        checkpoint=checkpoint,
    )
    print("Loaded checkpoint:", checkpoint)
    if MODEL_FACTORY_ERRORS:
        print("Model factory attempts that were skipped:", MODEL_FACTORY_ERRORS)

# Infer whether the historical CNN expects [B,F] and unsqueezes internally or
# expects an explicit [B,1,F] tensor. This decision is frozen for the notebook.
MODEL_INPUT_MODE = None
probe_out = None
candidate_inputs = [("raw", raw_example)]
if raw_example.ndim == 2:
    candidate_inputs.append(("unsqueeze_channel", raw_example.unsqueeze(1)))
errors = {}
MODEL.eval()
for mode, candidate in candidate_inputs:
    try:
        with torch.no_grad():
            probe_out = MODEL(candidate)
        MODEL_INPUT_MODE = mode
        EXAMPLE_INPUT = candidate
        break
    except Exception as exc:
        errors[mode] = repr(exc)

if MODEL_INPUT_MODE is None:
    raise RuntimeError(
        "Could not infer the CNN input convention. Set EXAMPLE_INPUT and "
        "MODEL_INPUT manually. Attempts: " + json.dumps(errors, indent=2)
    )

def MODEL_INPUT(x):
    if MODEL_INPUT_MODE == "unsqueeze_channel" and x.ndim == 2:
        return x.unsqueeze(1)
    return x

if probe_out.shape[1] != len(CLASS_NAMES):
    raise RuntimeError(
        f"Model outputs {probe_out.shape[1]} classes but CLASS_NAMES has "
        f"{len(CLASS_NAMES)} entries. Supply the exact label-encoder order."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(DEVICE)
EXAMPLE_INPUT = EXAMPLE_INPUT.to(DEVICE)
print("Device:", DEVICE)
print("Model:", type(MODEL).__name__)
print("Classes:", len(CLASS_NAMES))
print("Input convention:", MODEL_INPUT_MODE, tuple(EXAMPLE_INPUT.shape))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from src.saber.adapters import collect_logits
from src.saber.taxonomy import (
    ciciot2023_taxonomy,
    DEFAULT_COST_PROFILES,
    write_cost_profiles,
)

OUT = OUTPUT_ROOT / "14_risk_graph"
OUT.mkdir(parents=True, exist_ok=True)

taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
taxonomy_df = pd.DataFrame(taxonomy.to_records())
taxonomy_df.to_csv(OUT / "ciciot2023_taxonomy.csv", index=False)
write_cost_profiles(str(OUT / "cost_profiles.json"))

print(taxonomy_df.groupby(["binary_label", "family"]).size())
print("\nCost profiles:")
for name, profile in DEFAULT_COST_PROFILES.items():
    print(name, profile.as_dict())

## 2. Extract validation logits

The output can be cached and resumed. If the validation loader contains a third
field (part-file, capture group, or another provenance environment), it is used
to downweight graph edges that are unstable across environments.

In [ ]:
VAL_CACHE = OUT / "validation_teacher_outputs.npz"

if VAL_CACHE.exists():
    cached = np.load(VAL_CACHE, allow_pickle=True)
    VAL_LOGITS = cached["logits"]
    VAL_LABELS = cached["labels"].astype(np.int64)
    VAL_ENV = cached["environments"] if "environments" in cached.files else None
    if VAL_ENV is not None and VAL_ENV.dtype == object and len(VAL_ENV) == 1 and VAL_ENV[0] is None:
        VAL_ENV = None
    print("Loaded cached validation outputs:", VAL_CACHE)
else:
    VAL_LOGITS, VAL_LABELS, VAL_ENV = collect_logits(
        MODEL, VAL_LOADER, device=DEVICE, input_transform=MODEL_INPUT
    )
    payload = {"logits": VAL_LOGITS, "labels": VAL_LABELS}
    if VAL_ENV is not None:
        payload["environments"] = VAL_ENV
    np.savez_compressed(VAL_CACHE, **payload)
    print("Cached validation outputs:", VAL_CACHE)

assert VAL_LOGITS.shape == (len(VAL_LABELS), taxonomy.n_classes)
print("Validation logits:", VAL_LOGITS.shape)
print("Environment field:", None if VAL_ENV is None else (VAL_ENV.shape, len(np.unique(VAL_ENV))))

## 3. Build one graph per cost profile and a robust graph

For each true source class, candidates are ranked using validation confusion and
small teacher pairwise margins. Directed transition cost is then applied.
`cvar` aggregation protects edges that are important under the upper tail of
the three plausible operational profiles.

In [ ]:
from src.saber.risk_graph import (
    aggregate_robust_edge_weights,
    build_alert_semantic_vulnerability_graph,
    merge_cost_profile_graphs,
    save_graph_bundle,
)

graph_cfg = SABER_CFG["risk_graph"]
profile_graphs = []
for profile_name, profile in DEFAULT_COST_PROFILES.items():
    graph = build_alert_semantic_vulnerability_graph(
        VAL_LOGITS,
        VAL_LABELS,
        taxonomy,
        profile,
        top_k=int(graph_cfg["top_k_competitors"]),
        temperature=float(graph_cfg["temperature"]),
        confusion_weight=float(graph_cfg["confusion_weight"]),
        margin_weight=float(graph_cfg["margin_weight"]),
        min_class_support=int(graph_cfg["min_class_support"]),
        environments=VAL_ENV,
    )
    profile_graphs.append(graph)

edges_by_profile = merge_cost_profile_graphs(profile_graphs)
robust_graph = aggregate_robust_edge_weights(
    edges_by_profile,
    method=str(graph_cfg["robust_aggregation"]),
    q=float(graph_cfg["robust_cvar_q"]),
)

metadata = {
    "dataset": taxonomy.dataset,
    "n_validation": int(len(VAL_LABELS)),
    "n_classes": taxonomy.n_classes,
    "families": list(taxonomy.families),
    "cost_profiles": list(DEFAULT_COST_PROFILES),
    "risk_graph_config": graph_cfg,
    "environment_count": None if VAL_ENV is None else int(len(np.unique(VAL_ENV))),
}
save_graph_bundle(OUT, edges_by_profile, robust_graph, metadata)

display(robust_graph.head(20))

## 4. Audit graph coverage and edge semantics

The graph is invalid if labels remain unmapped, if important source classes have
no edges, or if one transition family dominates only because of a coding error.
Inspect these tables before accepting the graph.

In [ ]:
coverage = (
    robust_graph.groupby(["source_class", "source_family"], as_index=False)
    .agg(
        edges=("target_class", "count"),
        total_weight=("robust_weight", "sum"),
        maximum_edge_weight=("robust_weight", "max"),
    )
    .sort_values("total_weight", ascending=False)
)
transition_summary = (
    robust_graph.groupby("transition_type", as_index=False)
    .agg(edges=("target_class", "count"), weight=("robust_weight", "sum"))
    .sort_values("weight", ascending=False)
)
family_flow = (
    robust_graph.groupby(["source_family", "target_family"], as_index=False)
    .agg(edges=("target_class", "count"), weight=("robust_weight", "sum"))
    .sort_values("weight", ascending=False)
)

coverage.to_csv(OUT / "asvg_source_coverage.csv", index=False)
transition_summary.to_csv(OUT / "asvg_transition_summary.csv", index=False)
family_flow.to_csv(OUT / "asvg_family_flow.csv", index=False)

display(coverage.head(20))
display(transition_summary)
display(family_flow.head(20))

assert robust_graph["source_index"].nunique() >= taxonomy.n_classes - 1, (
    "Unexpectedly low graph source coverage. Inspect class support and labels."
)
assert np.isclose(robust_graph["robust_weight"].sum(), 1.0)

## 5. Visual diagnostics

The full graph can be visually dense. The first plot shows only the strongest
edges; the second shows the operational transition-weight distribution.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

top_edges = robust_graph.nlargest(min(24, len(robust_graph)), "robust_weight")
G = nx.DiGraph()
for row in top_edges.itertuples():
    G.add_edge(
        row.source_class,
        row.target_class,
        weight=float(row.robust_weight),
        transition=row.transition_type,
    )

plt.figure(figsize=(13, 9))
pos = nx.spring_layout(G, seed=42, k=1.2)
widths = [
    1.0 + 12.0 * G[u][v]["weight"] / max(top_edges["robust_weight"].max(), 1e-12)
    for u, v in G.edges()
]
nx.draw_networkx_nodes(G, pos, node_size=1100)
nx.draw_networkx_labels(G, pos, font_size=7)
nx.draw_networkx_edges(G, pos, width=widths, arrows=True, arrowsize=15, alpha=0.75)
plt.title("SABER-IDS validation-only Alert-Semantic Vulnerability Graph")
plt.axis("off")
plt.tight_layout()
plt.savefig(OUT / "asvg_top_edges.png", dpi=250, bbox_inches="tight")
plt.show()

ax = transition_summary.plot(
    x="transition_type", y="weight", kind="bar", legend=False, figsize=(9, 4)
)
ax.set_ylabel("Robust graph weight")
ax.set_xlabel("")
ax.set_title("Operational transition composition of the robust graph")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUT / "asvg_transition_weights.png", dpi=250, bbox_inches="tight")
plt.show()

## 6. Freeze the graph artefacts

Review the CSVs before proceeding. Once Notebook 16 starts score validation,
do not change the taxonomy, cost profiles, or graph hyperparameters without
creating a new version and recording the reason.

In [ ]:
import subprocess
from src.saber.adapters import save_run_manifest

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip()
except Exception:
    git_commit = None

artefacts = [
    OUT / "ciciot2023_taxonomy.csv",
    OUT / "cost_profiles.json",
    OUT / "asvg_edges_by_cost_profile.csv",
    OUT / "asvg_edges_robust.csv",
    OUT / "asvg_source_coverage.csv",
    OUT / "asvg_transition_summary.csv",
    OUT / "asvg_family_flow.csv",
    OUT / "asvg_top_edges.png",
    OUT / "asvg_transition_weights.png",
]
save_run_manifest(
    OUT / "manifest.json",
    notebook="14_build_semantic_hierarchy_and_risk_graph.ipynb",
    config=SABER_CFG,
    artifacts=artefacts,
    git_commit=git_commit,
)
print("Notebook 14 complete. Review:", OUT)